### **<u> I. Import des données <u>**

In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(123)

In [44]:
# Url pour accéder au fichier CSV sur sspcloud
url = "https://minio.lab.sspcloud.fr/guigz/donnees_GBD_uniques.csv"

# Lecture directe du CSV
df_GBD = pd.read_csv(url, sep=";", encoding="latin-1", decimal=",")

df_GBD.head()

,population_group,measure,location_id,location_name,sex,age_name,cause_id,cause_name,metric_id,metric_name,year,value,upper_bound,lower_bound
0,All Population,Deaths,71,Australia,Both,All ages,294,All causes,1,Number,1980,108265.029600,108882.768100,107497.458600
1,All Population,Deaths,71,Australia,Both,All ages,491,Cardiovascular diseases,1,Number,1980,53252.643510,54951.178210,50203.750100
2,All Population,Deaths,71,Australia,Both,All ages,521,Cirrhosis and other chronic liver diseases,1,Number,1980,1390.286718,1469.238444,1314.555387
3,All Population,Deaths,71,Australia,Both,All ages,945,Conflict and terrorism,1,Number,1980,3.999771,4.397038,3.641975
4,All Population,Deaths,71,Australia,Both,All ages,724,Interpersonal violence,1,Number,1980,297.285701,330.648576,270.154420


### **<u> II. Modification du dataframe GBD pour faire correspondre les causes_GBD avec les causes_2006 (on passe de 14 causes à 9 causes) <u>**

#### <u> 1. Création d'un dataframe qui ne comporte que les info "utiles" <u>

In [45]:
# création d'un dataframe qui ne comporte que les infos utiles == qui different d'un pays à l'autre, etc 

df_GBD_modifie = df_GBD[["location_id", "location_name", "cause_id", "cause_name", "metric_id", "metric_name", "year", "value","upper_bound", "lower_bound"]]

# conversion du texte en valeurs pour value, upper_bound, lower_bound
df_GBD_modifie["value"] = pd.to_numeric(df_GBD_modifie["value"])
df_GBD_modifie["upper_bound"] = pd.to_numeric(df_GBD_modifie["upper_bound"])
df_GBD_modifie["lower_bound"] = pd.to_numeric(df_GBD_modifie["lower_bound"])

# ajout des écarts-types et variances pour après calculer les nouvelles upper_bound et lower_bound

df_GBD_modifie["ecart_type"] = (df_GBD_modifie["upper_bound"] - df_GBD_modifie["lower_bound"]) / (2 * 1.96)
df_GBD_modifie["var"] = df_GBD_modifie["ecart_type"]**2

df_GBD_modifie.head()

,location_id,location_name,cause_id,cause_name,metric_id,metric_name,year,value,upper_bound,lower_bound,ecart_type,var
0,71,Australia,294,All causes,1,Number,1980,108265.029600,108882.768100,107497.458600,353.395281,1.248882e+05
1,71,Australia,491,Cardiovascular diseases,1,Number,1980,53252.643510,54951.178210,50203.750100,1211.078599,1.466711e+06
2,71,Australia,521,Cirrhosis and other chronic liver diseases,1,Number,1980,1390.286718,1469.238444,1314.555387,39.459964,1.557089e+03
3,71,Australia,945,Conflict and terrorism,1,Number,1980,3.999771,4.397038,3.641975,0.192618,3.710169e-02
4,71,Australia,724,Interpersonal violence,1,Number,1980,297.285701,330.648576,270.154420,15.432183,2.381523e+02


#### <u> 2. Création du dataframe avec les causes générales (de 2006) aggrégées <u>

Introduction des paramètres de fonction (metric et causes_generales)

In [46]:
# introduction des métriques

metric = ["Number", "Percent", "Rate"]

cause_2 = [1012, 459, 489] # !! pour tester que tout marche bien, à enlever à la fin pour le rendu !!!

# introduction des nouvelles causes 

causes_generales = [["Malignant neoplasm", 2, [1012, 459, 489]], ["Motor vehicle accidents", 6, [689, 688]], ["Homicide", 9, [724, 945,854]]]

Fonction qui prend en argument un dataframe, la liste des causes associées à une cause générale et renvoie une liste de trois dataframe (un pour chaque métrique) qui contient le pays, l'année, les valeurs aggrégées pour cette cause générale et la métrique

In [47]:
def dataframe_par_metrique(df, cause, metric) :
    L = [] # initialisation de la liste
    for i in range(1,4) : # pour chaque métrique
        df_filtre_i = df[(df["cause_id"].isin(cause)) & (df["metric_id"] == i)] # filtration
        df_aggreg_i = df_filtre_i.groupby(["location_id","location_name", "year"], as_index=False)["value"].sum() # aggrégation
        df_aggreg_i = df_aggreg_i.assign(
            metric_id = i,
            metric_name = metric[i-1]
        ) # ajout du nom et le numéro de la métrique  
        L.append(df_aggreg_i)
    return(L)

dataframe_par_metrique(df_GBD_modifie,cause_2, metric)[2].head()

,location_id,location_name,year,value,metric_id,metric_name
0,47,Czechia,1980,9.186061,3,Rate
1,47,Czechia,1981,9.108751,3,Rate
2,47,Czechia,1982,9.066288,3,Rate
3,47,Czechia,1983,9.237296,3,Rate
4,47,Czechia,1984,9.586514,3,Rate


Fonction qui prend en argument un dataframe, une liste de trois dataframe et la liste des causes associées à une cause générale, et renvoie la liste des trois dataframe en ayant ajouté le lower_bound et le higher_bound

In [48]:
def ajout_bounds(df, L, cause) :
    N = []
    for i in range(1,4) : 
        df_filtre_i = df[(df["cause_id"].isin(cause)) & (df["metric_id"] == i)] # filtration
        df_aggreg_var_i = df_filtre_i.groupby(["location_id","location_name", "year"], as_index=False)["var"].sum() # somme des variances
        df_aggreg_var_i["ecart_type"] = np.sqrt(df_aggreg_var_i["var"]) # écart-type
        
        M = df_aggreg_var_i["ecart_type"].tolist()
        M = [1.96 * x for x in M]

        L[i-1]["lower_bound"] = L[i-1]["value"] - M # calcul du lower_bound
        L[i-1]["upper_bound"] = L[i-1]["value"] + M # calcul du upper_bound
        N.append(L[i-1])
    return(N)

ajout_bounds(df_GBD_modifie,dataframe_par_metrique(df_GBD_modifie,cause_2, metric),cause_2)[0].tail()

,location_id,location_name,year,value,metric_id,metric_name,lower_bound,upper_bound
1667,155,Türkiye,2019,2349.845349,1,Number,1944.931638,2754.759060
1668,155,Türkiye,2020,2448.668022,1,Number,2016.465770,2880.870273
1669,155,Türkiye,2021,2509.954887,1,Number,2044.363282,2975.546492
1670,155,Türkiye,2022,2695.978660,1,Number,2204.225458,3187.731862
1671,155,Türkiye,2023,2835.548990,1,Number,2282.044321,3389.053659


Fonction qui prend en argument une liste de dataframe et renvoie leur concaténation

In [49]:
def concatenation(N) :
    df_concatene = pd.concat(N, ignore_index=True)
    return (df_concatene)

concatenation(ajout_bounds(df_GBD_modifie,dataframe_par_metrique(df_GBD_modifie,cause_2, metric),cause_2))


,location_id,location_name,year,value,metric_id,metric_name,lower_bound,upper_bound
0,47,Czechia,1980,943.567925,1,Number,878.273547,1008.862303
1,47,Czechia,1981,936.655643,1,Number,874.307188,999.004099
2,47,Czechia,1982,932.900040,1,Number,872.915908,992.884171
3,47,Czechia,1983,950.685882,1,Number,891.336740,1010.035024
4,47,Czechia,1984,986.545496,1,Number,928.614820,1044.476172
...,...,...,...,...,...,...,...,...
5011,155,Türkiye,2019,2.830499,3,Rate,2.342761,3.318236
5012,155,Türkiye,2020,2.923271,3,Rate,2.407299,3.439243
5013,155,Türkiye,2021,2.970430,3,Rate,2.419421,3.521439
5014,155,Türkiye,2022,3.161625,3,Rate,2.584937,3.738313


Fonction qui prend en argument un dataframe, une cause générale (liste avec nom de la cause, numéro associé, et liste de numéro des sous-causes) et renvoie le même dataframe en ayant rajouté le nom et le numéro de la cause générale

In [50]:
def ajout_cause (df, cause_generale) :
    df = df.assign(
    cause_id = cause_generale[1],
    cause_name = cause_generale[0],
    )
    return(df)

ajout_cause(concatenation(ajout_bounds(df_GBD_modifie,dataframe_par_metrique(df_GBD_modifie,cause_2, metric),cause_2)), causes_generales[0])


,location_id,location_name,year,value,metric_id,metric_name,lower_bound,upper_bound,cause_id,cause_name
0,47,Czechia,1980,943.567925,1,Number,878.273547,1008.862303,2,Malignant neoplasm
1,47,Czechia,1981,936.655643,1,Number,874.307188,999.004099,2,Malignant neoplasm
2,47,Czechia,1982,932.900040,1,Number,872.915908,992.884171,2,Malignant neoplasm
3,47,Czechia,1983,950.685882,1,Number,891.336740,1010.035024,2,Malignant neoplasm
4,47,Czechia,1984,986.545496,1,Number,928.614820,1044.476172,2,Malignant neoplasm
...,...,...,...,...,...,...,...,...,...,...
5011,155,Türkiye,2019,2.830499,3,Rate,2.342761,3.318236,2,Malignant neoplasm
5012,155,Türkiye,2020,2.923271,3,Rate,2.407299,3.439243,2,Malignant neoplasm
5013,155,Türkiye,2021,2.970430,3,Rate,2.419421,3.521439,2,Malignant neoplasm
5014,155,Türkiye,2022,3.161625,3,Rate,2.584937,3.738313,2,Malignant neoplasm


Fonction qui prend en argument un dataframe, une liste de causes générales, une métrique et renvoie le dataframe qui aggrège ce dernier par rapport aux causes générales

In [51]:
def aggrege(df, causes_generales, metric) :
    A = [] # future liste des data frame pour chaque cause générale
    for j in range(len(causes_generales)) : # pour chaque cause
        L = dataframe_par_metrique(df, causes_generales[j][2], metric) # aggrégation des sous_causes par métrique
        N = ajout_bounds(df, L, causes_generales[j][2]) # ajout des upper_bound et lower_bound
        dataframe = concatenation(N) # concaténation des trois dataframe (1 pour chaque métrique)
        A.append(ajout_cause (dataframe, causes_generales[j])) # ajout du nouveau nom de la cause
    return(concatenation(A)) # concaténation des dataframe de chaque cause 

df_modif = aggrege(df_GBD_modifie, causes_generales, metric)

df_modif.head(10)


,location_id,location_name,year,value,metric_id,metric_name,lower_bound,upper_bound,cause_id,cause_name
0,47,Czechia,1980,943.567925,1,Number,878.273547,1008.862303,2,Malignant neoplasm
1,47,Czechia,1981,936.655643,1,Number,874.307188,999.004099,2,Malignant neoplasm
2,47,Czechia,1982,932.900040,1,Number,872.915908,992.884171,2,Malignant neoplasm
3,47,Czechia,1983,950.685882,1,Number,891.336740,1010.035024,2,Malignant neoplasm
4,47,Czechia,1984,986.545496,1,Number,928.614820,1044.476172,2,Malignant neoplasm
5,47,Czechia,1985,1113.419858,1,Number,1047.751206,1179.088509,2,Malignant neoplasm
6,47,Czechia,1986,1270.227815,1,Number,1196.549530,1343.906099,2,Malignant neoplasm
7,47,Czechia,1987,1272.280705,1,Number,1201.363761,1343.197648,2,Malignant neoplasm
8,47,Czechia,1988,1282.793881,1,Number,1214.784905,1350.802857,2,Malignant neoplasm
9,47,Czechia,1989,1303.958518,1,Number,1235.483238,1372.433797,2,Malignant neoplasm


### <u> 3. Création d'un dataframe qui contient les causes qui n'ont pas eu besoin d'être aggrégées <u>

In [52]:
# création d'un data frame qui contient les valeurs qui n'ont pas du être agrégées

numero_non_agg = [294,491,521,322,718,696] # numeros des causes non aggrégées 
df_sans_modif = df_GBD_modifie[df_GBD_modifie["cause_id"].isin(numero_non_agg)]

# renommer les noms (sauf pour all causes)

noms = [[491, "Major cardiovascular diseases"], [521, "Liver diseases, cirrhosis"], [322, "Pneumonia and influenza"], [718,"Suicide"], [696, "Other accidents"]]
for i in range(len(noms)) :
    df_sans_modif.loc[df_sans_modif["cause_id"] == noms[i][0], "cause_name"] = noms[i][1]

# renommer les numéros des causes

numero = [["All causes", 1], ["Major cardiovascular diseases", 3], ["Liver diseases, cirrhosis", 4], ["Pneumonia and influenza", 5], ["Suicide", 7], ["Other accidents", 8]]
for i in range(len(numero)) :
    df_sans_modif.loc[df_sans_modif["cause_name"] == numero[i][0], "cause_id"] = numero[i][1]

# suppression de la variance et de l'écart-type
df_sans_modif = df_sans_modif.drop(columns=["ecart_type", "var"])

df_sans_modif.tail(10)

,location_id,location_name,cause_id,cause_name,metric_id,metric_name,year,value,upper_bound,lower_bound
70198,102,United States of America,4,"Liver diseases, cirrhosis",3,Rate,2022,21.958110,23.459380,20.337288
70201,102,United States of America,5,Pneumonia and influenza,3,Rate,2022,19.037326,21.950451,16.086608
70207,102,United States of America,7,Suicide,3,Rate,2022,16.066228,17.474083,14.670744
70209,102,United States of America,8,Other accidents,3,Rate,2022,26.654055,29.180480,23.585686
70210,102,United States of America,1,All causes,3,Rate,2023,921.742696,922.487018,920.993240
70211,102,United States of America,3,Major cardiovascular diseases,3,Rate,2023,268.551272,288.466433,233.253568
70212,102,United States of America,4,"Liver diseases, cirrhosis",3,Rate,2023,21.597735,23.211950,19.887877
70215,102,United States of America,5,Pneumonia and influenza,3,Rate,2023,18.972363,21.894608,16.044547
70221,102,United States of America,7,Suicide,3,Rate,2023,15.281942,16.705880,13.797355
70223,102,United States of America,8,Other accidents,3,Rate,2023,26.463833,28.945239,23.261633


#### <u> 4. Dataframe GBD final <u>

On obtient au final un dataframe contenant 45144 lignes (9 causes * 44 années * 3 mesures * 38 pays) et 7 colonnes. 

In [ ]:
# concaténation des deux dataframe (celui avec les causes aggrégées et celui avec les autres)

df_GBD_final = concatenation([df_sans_modif, df_modif]) 

# renommer les location_id

liste_pays = df_GBD_final["location_name"].tolist() # convertir la colonne location_names en liste
liste_pays = list(dict.fromkeys(liste_pays))# ne garde qu'une occurrence
for pays in liste_pays :
    df_GBD_final.loc[df_GBD_final["location_name"] == pays, "location_id"] = liste_pays.index(pays) + 1 # associer chaque pays à son numéro (ordre alphabétique)

# réarrangement 

df_GBD_final = df_GBD_final.sort_values(by=["location_name","metric_id", "year", "cause_id"]).reset_index(drop=True)


df_GBD_final.head(1188)
#df_GBD_final.tail(12)

,location_id,location_name,cause_id,cause_name,metric_id,metric_name,year,value,upper_bound,lower_bound
0,1,Australia,1,All causes,1,Number,1980,108265.029600,108882.768100,107497.458600
1,1,Australia,2,Malignant neoplasm,1,Number,1980,1109.779639,1159.646745,1059.912533
2,1,Australia,3,Major cardiovascular diseases,1,Number,1980,53252.643510,54951.178210,50203.750100
3,1,Australia,4,"Liver diseases, cirrhosis",1,Number,1980,1390.286718,1469.238444,1314.555387
4,1,Australia,5,Pneumonia and influenza,1,Number,1980,1818.331630,1921.866841,1690.713494
...,...,...,...,...,...,...,...,...,...,...
1183,1,Australia,5,Pneumonia and influenza,3,Rate,2023,14.152036,15.788212,11.511009
1184,1,Australia,6,Motor vehicle accidents,3,Rate,2023,10.983947,11.706293,10.261600
1185,1,Australia,7,Suicide,3,Rate,2023,12.288987,13.068551,11.599252
1186,1,Australia,8,Other accidents,3,Rate,2023,24.905355,27.991055,20.710593


## Statistiques descriptives

In [54]:
#Fonction qui permet de créer un sous dataframe à partir d'un datafram original, d'une métrique, d'une cause et d'un pays
#Cela permet de ne garder que l'évolution d'une cause précise pour un pays et une métrique donnés
def sous_dataframe(df, metric_id, cause_id, location_id) :
    df_sous = df[(df["metric_id"] == metric_id) & (df["cause_id"] == cause_id) & (df["location_id"] == location_id)]
    return(df_sous)

Création d'un tableau de correspondance entre les indices et nom des pays, plus lisible.

In [55]:

df_location = df_GBD_final[(df_GBD_final["metric_id"] == 3) & (df_GBD_final["cause_id"] == 1) & (df_GBD_final["year"] == 2023)][["location_id", "location_name"]]
df_location

#Pour faire la même chose avec les métriques taper df_GBD_final.head(3)
#Pour faire la même chose avec les causes taper df_GBD_final.head(9)

,location_id,location_name
1179,1,Australia
2367,2,Austria
3555,3,Belgium
4743,4,Canada
5931,5,Chile
7119,6,Colombia
8307,7,Costa Rica
9495,8,Czechia
10683,9,Denmark
11871,10,Estonia


In [56]:
df_allcauses_france_rate = sous_dataframe(df_GBD_final, 1, 12, 3) # exemple pour toutes le taux de toutes les causes en France
df_allcauses_france_rate

,location_id,location_name,cause_id,cause_name,metric_id,metric_name,year,value,upper_bound,lower_bound
